# Follow-up experiments (v2.1)
Re-runs only two parts of v2:
1. Fine-tuned BERT for NFR **subtype** classification with an adequate training budget (v2 stopped after too few steps on small data).
2. Per-FR **top-k** link selection, because a single global threshold τ never beat accepting all pairs.

**Run:** T4 GPU → Run all → upload the same CSV. About 25–35 minutes. Downloads `results_v21_bundle.zip`.
# (original v2 header below)

**What this notebook does**
* **D1** – your e-commerce corpus (LLM-generated, expert-reviewed). You upload `Labeled_Requirements_Dataset.csv`.
* **D2** – PROMISE_exp (969 requirements, 15 projects), a public benchmark. Downloaded automatically.
* **D3** – PROMISE NFR manual FR→NFR trace matrices (10 projects), giving *real ground truth* for linking. Downloaded automatically.

Experiments: held-out evaluation with McNemar tests, repeated 5×2 cross-validation with Alpaydin's F-test,
Wilcoxon tests and effect sizes, leave-one-project-out (cross-project) evaluation, cross-dataset transfer
(synthetic → real and back), ablations, calibration, lexical vs. semantic vs. hybrid linking, and efficiency.

**Run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Upload the CSV when asked.
Total time is roughly 45–90 minutes with the GPU (set `RUN_BERT = False` for a ~15-minute run without BERT).
At the end `results_v2_bundle.zip` downloads: all tables (CSV), figures (300-dpi PNG) and `results_v2.json`.

In [ ]:
# ---------- 0. Setup ----------
!pip -q install imbalanced-learn==0.12.4 sentence-transformers rank_bm25 psutil

In [ ]:
import os, re, json, time, random, platform, warnings, zipfile, glob, subprocess, itertools
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import psutil, sklearn, imblearn, scipy
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RepeatedStratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix,
                             average_precision_score, brier_score_loss)
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
warnings.filterwarnings("ignore")

# ---------------- configuration ----------------
SEED = 42
RUN_NEURAL = True          # SBERT embeddings (classification + linking)
RUN_BERT = True            # fine-tuned BERT classifier (needs GPU for reasonable speed)
BERT_IN_CV = True          # include BERT in the 5x2 CV (FR/NFR task only); adds ~20 fine-tuning runs
SBERT_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BERT_NAME = "bert-base-uncased"
BERT_EPOCHS, BERT_LR, BERT_BS, BERT_MAXLEN = 4, 3e-5, 16, 64
BERT_MIN_STEPS = 600       # at least 600 optimisation steps, so small training sets get enough epochs
PROMISE_REPO = "https://github.com/AleksandarMitrevski/se-requirements-classification"
PROMISE_COMMIT = "1f5dc4501a1956f21011083594d925bac49f198c"   # pinned for reproducibility
OUT = "results_v21"; os.makedirs(OUT, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception: pass
set_seed()
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9})
R = {"seed": SEED, "env": {"python": platform.python_version(), "sklearn": sklearn.__version__, "imblearn": imblearn.__version__,
                           "numpy": np.__version__, "scipy": scipy.__version__}}
def save_fig(name): plt.tight_layout(); plt.savefig(f"{OUT}/{name}.png", bbox_inches="tight"); plt.show(); plt.close()
def save_tab(df, name): df.to_csv(f"{OUT}/{name}.csv", index=False); print(f"\n== {name} ==\n", df.to_string(index=False))

In [ ]:
# ---------- 1. Preprocessing ----------
USE_LEMMATIZER = True
try:
    import nltk
    nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
    from nltk.stem import WordNetLemmatizer
    _lem = WordNetLemmatizer(); _lem.lemmatize("tests")
except Exception as e:
    print("WordNet unavailable -> lemmatisation disabled:", e); USE_LEMMATIZER = False
R["lemmatization_available"] = USE_LEMMATIZER
STOP = set("""i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself yourselves he him his himself she she's her hers herself it it's its itself they them their theirs themselves what which who whom this that that'll these those am is are was were be been being have has had having do does did doing a an the and but if or because as until while of at by for with about against between into through during before after above below to from up down in out on off over under again further then once here there when where why how all any both each few more most other some such no nor not only own same so than too very s t can will just don don't should should've now d ll m o re ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn hasn't haven haven't isn isn't ma mightn mightn't mustn mustn't needn needn't shan shan't shouldn shouldn't wasn wasn't weren weren't won won't wouldn wouldn't""".split())
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9]+")
def preprocess(text, lemmatize=True, stopwords=True):
    toks = TOKEN_RE.findall(str(text))
    if stopwords: toks = [t for t in toks if t.lower() not in STOP]
    toks = [t for t in toks if len(t) > 1]
    if lemmatize and USE_LEMMATIZER: toks = [_lem.lemmatize(_lem.lemmatize(t.lower(), "n"), "v") for t in toks]
    return " ".join(t.lower() for t in toks)

def jaccard_matrix(a, b):
    cv = CountVectorizer(binary=True, token_pattern=r"\S+").fit(list(a) + list(b))
    A = cv.transform(a).astype(np.float32); B = cv.transform(b).astype(np.float32)
    inter = (A @ B.T).toarray(); sa = np.asarray(A.sum(1)).ravel()[:, None]; sb = np.asarray(B.sum(1)).ravel()[None, :]
    return inter / np.maximum(sa + sb - inter, 1)

def clean_corpus(df, name, min_sub=10):
    """df columns: text, type (FR/NFR), sub, (module), (project). Removes empties, exact and near duplicates, rare subtypes."""
    n0 = len(df); df = df.copy()
    df["clean"] = df["text"].map(preprocess)
    df["clean_nolem"] = df["text"].map(lambda t: preprocess(t, lemmatize=False))
    df["clean_nostop"] = df["text"].map(lambda t: preprocess(t, stopwords=False))
    df = df[df["clean"].str.len() > 0]
    df["_norm"] = df["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip(" .")
    df = df.drop_duplicates("_norm").reset_index(drop=True); n_exact = n0 - len(df)
    J = jaccard_matrix(df["clean"], df["clean"]); np.fill_diagonal(J, 0); drop = set()
    for i in range(len(df)):
        if i in drop: continue
        drop.update(j for j in np.where(J[i] >= 0.95)[0] if j > i)
    df = df.drop(index=sorted(drop)).reset_index(drop=True)
    sc = df[df.type == "NFR"]["sub"].value_counts(); rare = sc[sc < min_sub].index.tolist()
    n_rare = int(df["sub"].isin(rare).sum()); df = df[~df["sub"].isin(rare)].reset_index(drop=True)
    info = {"raw": n0, "exact_dups": n_exact, "near_dups": len(drop), "rare_subtypes_removed": rare, "rare_removed": n_rare,
            "final": len(df), "type_counts": df.type.value_counts().to_dict(), "sub_counts": df[df.type == "NFR"]["sub"].value_counts().to_dict()}
    print(name, json.dumps(info)); return df, info

In [ ]:
# ---------- 2. Data ----------
# D1: user corpus
CSV = "Labeled_Requirements_Dataset.csv"
if not os.path.exists(CSV):
    from google.colab import files
    up = files.upload(); CSV = list(up.keys())[0]
raw1 = pd.read_csv(CSV); raw1.columns = [c.strip() for c in raw1.columns]
d1 = pd.DataFrame({"text": raw1["Requirement Text"].astype(str).str.strip(), "type": raw1["Type"].astype(str).str.strip(),
                   "sub": raw1["Subcategory"].astype(str).str.strip(),
                   "module": raw1["Source/Module"].astype(object).where(raw1["Source/Module"].notna(), "Unknown").astype(str).str.strip()})
d1, R["D1_info"] = clean_corpus(d1, "D1"); R["D1_info"]["n_modules"] = int(d1.module.nunique())

# D2 + D3: PROMISE_exp and PROMISE NFR traces (pinned commit)
if not os.path.exists("promise_repo"):
    subprocess.run(["git", "clone", "-q", PROMISE_REPO, "promise_repo"], check=True)
    subprocess.run(["git", "-C", "promise_repo", "checkout", "-q", PROMISE_COMMIT], check=True)
CODES = {"F": "Functional", "A": "Availability", "FT": "Fault Tolerance", "L": "Legal", "LF": "Look & Feel", "MN": "Maintainability",
         "O": "Operability", "PE": "Performance", "PO": "Portability", "SC": "Scalability", "SE": "Security", "US": "Usability"}
raw2 = pd.read_csv("promise_repo/1-exploratory-analysis/data/PROMISE_exp.csv")
d2 = pd.DataFrame({"text": raw2["RequirementText"].astype(str).str.strip(), "type": np.where(raw2["_class_"] == "F", "FR", "NFR"),
                   "sub": raw2["_class_"].map(CODES), "project": raw2["ProjectID"].astype(int)})
d2, R["D2_info"] = clean_corpus(d2, "D2")

def read_arff_rows(path):
    rows, data = [], False
    for l in open(path, encoding="latin-1"):
        l = l.strip()
        if l.lower().startswith("@data"): data = True; continue
        if data and l and not l.startswith("%"): rows.append(l)
    return rows
base = "promise_repo/0-datasets/tera-PROMISE NFR"
nfr_rows = []
for l in read_arff_rows(f"{base}/nfr.arff"):
    m = re.match(r"^(\d+),'(.*)',(\w+)$", l)
    if m: nfr_rows.append((int(m[1]), m[2].replace("\\'", "'"), m[3]))
promise_nfr = pd.DataFrame(nfr_rows, columns=["project", "text", "label"])
D3 = {}
for p in range(1, 11):
    proj = promise_nfr[promise_nfr.project == p].reset_index(drop=True)          # R1..Rn = order within project
    lines = read_arff_rows(f"{base}/P(1-10)_NFR_FR_trace/P{p}_NFR_FR_trace.arff")
    hdr = open(f"{base}/P(1-10)_NFR_FR_trace/P{p}_NFR_FR_trace.arff", encoding="latin-1").read()
    n_nfr = len(re.findall(r"@attribute\s+NFR\d+", hdr))
    nfr_idx = [i for i in range(len(proj)) if proj.label[i] != "F"][:n_nfr]
    fr_ids, G = [], []
    for l in lines:
        parts = [x.strip() for x in l.split(",")]
        fr_ids.append(int(parts[0][1:]) - 1); G.append([int(x) for x in parts[1:1 + n_nfr]])
    D3[p] = {"fr": proj.text[fr_ids].tolist(), "nfr": proj.text[nfr_idx].tolist(), "G": np.array(G, dtype=bool),
             "fr_types_ok": bool((proj.label[fr_ids] == "F").all())}
R["D3_info"] = {p: {"n_fr": len(v["fr"]), "n_nfr": len(v["nfr"]), "links": int(v["G"].sum()), "density": float(v["G"].mean()),
                    "fr_types_ok": v["fr_types_ok"]} for p, v in D3.items()}
print(pd.DataFrame(R["D3_info"]).T)

In [ ]:
# ---------- 3. Representations and models ----------
def make_vec(ngram=(1, 2)): return TfidfVectorizer(ngram_range=ngram, min_df=2, sublinear_tf=True, norm="l2", token_pattern=r"\S+")
# note: min_df=2 is used for all datasets so the smaller PROMISE_exp keeps enough vocabulary
SBERT = None; EMB = {}
if RUN_NEURAL:
    from sentence_transformers import SentenceTransformer
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; print("Device:", DEVICE)
    SBERT = SentenceTransformer(SBERT_NAME, device=DEVICE)
    R["env"]["device"] = DEVICE
def embed(texts):
    key = tuple(texts)
    if key not in EMB: EMB[key] = SBERT.encode(list(texts), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    return EMB[key]

cv3 = lambda: StratifiedKFold(3, shuffle=True, random_state=SEED)
class TfidfModel:
    """TF-IDF features + a classical learner. text_col chooses the preprocessing variant (ablations)."""
    def __init__(self, kind, text_col="clean", ngram=(1, 2), oversample=True, calibrate="isotonic"):
        self.kind, self.text_col, self.ngram, self.oversample, self.calibrate = kind, text_col, ngram, oversample, calibrate
    def fit(self, df, y):
        self.vec = make_vec(self.ngram).fit(df[self.text_col]); X = self.vec.transform(df[self.text_col])
        steps = [("ros", RandomOverSampler(random_state=SEED))] if self.oversample else []
        if self.kind == "linsvc": est, grid = LinearSVC(max_iter=20000, random_state=SEED), {"m__C": [0.03, 0.1, 0.3, 1, 3]}
        elif self.kind == "lr": est, grid = LogisticRegression(max_iter=5000, class_weight="balanced"), {"m__C": [1, 10, 100]}
        elif self.kind == "rbf": est, grid = SVC(kernel="rbf", random_state=SEED), {"m__C": [1, 10]}
        elif self.kind == "rf": est, grid = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), {"m__max_features": ["sqrt"]}
        elif self.kind == "gb": est, grid = GradientBoostingClassifier(n_estimators=150, random_state=SEED), {"m__max_depth": [3]}
        elif self.kind == "nb": est, grid = ComplementNB(), {"m__alpha": [0.1, 0.5, 1.0]}
        k = max(2, min(3, pd.Series(y).value_counts().min()))
        gs = GridSearchCV(ImbPipeline(steps + [("m", est)]), grid, scoring="f1_macro",
                          cv=StratifiedKFold(k, shuffle=True, random_state=SEED), n_jobs=-1).fit(X, y)
        self.model = gs.best_estimator_; self.best = gs.best_params_
        self.binary = len(set(y)) == 2
        if self.kind == "linsvc" and self.binary and self.calibrate:
            Xr, yr = (RandomOverSampler(random_state=SEED).fit_resample(X, y) if self.oversample else (X, y))
            self.cal = CalibratedClassifierCV(LinearSVC(C=self.best["m__C"], max_iter=20000, random_state=SEED),
                                              method=self.calibrate, cv=3).fit(Xr, yr)
        else: self.cal = None
        return self
    def predict(self, df):
        X = self.vec.transform(df[self.text_col])
        return self.cal.predict(X) if self.cal is not None else self.model.predict(X)
    def proba(self, df):
        X = self.vec.transform(df[self.text_col])
        return self.cal.predict_proba(X)[:, 1] if self.cal is not None else None

class SbertLR:
    def fit(self, df, y):
        X = embed(df["text"].tolist())
        self.m = GridSearchCV(LogisticRegression(max_iter=5000, class_weight="balanced"), {"C": [0.3, 1, 3, 10]}, scoring="f1_macro",
                              cv=StratifiedKFold(max(2, min(3, pd.Series(y).value_counts().min())), shuffle=True, random_state=SEED), n_jobs=-1).fit(X, y)
        return self
    def predict(self, df): return self.m.predict(embed(df["text"].tolist()))
    def proba(self, df): return self.m.predict_proba(embed(df["text"].tolist()))[:, 1] if len(self.m.classes_) == 2 else None

class BertClf:
    def fit(self, df, y):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        set_seed()
        self.labels = sorted(set(y)); lid = {l: i for i, l in enumerate(self.labels)}; yy = np.array([lid[v] for v in y])
        self.tok = AutoTokenizer.from_pretrained(BERT_NAME)
        self.model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(self.labels), ignore_mismatched_sizes=True).to(DEVICE)
        enc = self.tok(df["text"].tolist(), truncation=True, padding="max_length", max_length=BERT_MAXLEN, return_tensors="pt")
        ds = torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(yy))
        dl = torch.utils.data.DataLoader(ds, batch_size=BERT_BS, shuffle=True, generator=torch.Generator().manual_seed(SEED))
        w = torch.tensor(len(yy) / (len(self.labels) * np.bincount(yy, minlength=len(self.labels)).clip(1)), dtype=torch.float).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=BERT_LR, weight_decay=0.01)
        epochs = max(BERT_EPOCHS, int(np.ceil(BERT_MIN_STEPS / len(dl)))); self.epochs_used = epochs
        steps = epochs * len(dl); sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, int(0.1 * steps))) * max(0.0, (steps - s) / steps))
        lossf = torch.nn.CrossEntropyLoss(weight=w); self.model.train()
        for _ in range(epochs):
            for ids, am, lab in dl:
                opt.zero_grad(); out = self.model(input_ids=ids.to(DEVICE), attention_mask=am.to(DEVICE))
                loss = lossf(out.logits, lab.to(DEVICE)); loss.backward(); torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0); opt.step(); sched.step()
        return self
    def _logits(self, df):
        import torch
        self.model.eval(); outs = []
        texts = df["text"].tolist()
        with torch.no_grad():
            for i in range(0, len(texts), 128):
                enc = self.tok(texts[i:i + 128], truncation=True, padding=True, max_length=BERT_MAXLEN, return_tensors="pt").to(DEVICE)
                outs.append(self.model(**enc).logits.float().cpu().numpy())
        return np.vstack(outs)
    def predict(self, df): return np.array(self.labels)[self._logits(df).argmax(1)]
    def proba(self, df):
        if len(self.labels) != 2: return None
        z = self._logits(df); e = np.exp(z - z.max(1, keepdims=True)); return (e / e.sum(1, keepdims=True))[:, self.labels.index("NFR")]

def model_zoo(task, for_cv=False):
    zoo = {"TF-IDF + Cal. LinearSVC (proposed)": lambda: TfidfModel("linsvc"),
           "TF-IDF + Logistic Regression": lambda: TfidfModel("lr"),
           "TF-IDF + Complement NB": lambda: TfidfModel("nb"),
           "TF-IDF + RBF-SVM": lambda: TfidfModel("rbf"),
           "TF-IDF + Random Forest": lambda: TfidfModel("rf")}
    if task == "binary": zoo["TF-IDF + Gradient Boosting"] = lambda: TfidfModel("gb")
    if RUN_NEURAL: zoo["SBERT + Logistic Regression"] = lambda: SbertLR()
    if RUN_BERT and RUN_NEURAL and (not for_cv or (BERT_IN_CV and task == "binary")): zoo["Fine-tuned BERT"] = lambda: BertClf()
    return zoo
PROPOSED = "TF-IDF + Cal. LinearSVC (proposed)"

def task_data(df, task):
    if task == "binary": return df, df["type"].values
    sub = df[df.type == "NFR"].reset_index(drop=True); return sub, sub["sub"].values

In [ ]:
# ---------- 4. Statistics helpers ----------
def boot_ci(fn, n, B=1000, seed=SEED):
    rng = np.random.default_rng(seed); v = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        try: v.append(fn(idx))
        except Exception: pass
    return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]
def mcnemar(y, a, b):
    ca, cb = (a == y), (b == y); n01 = int((ca & ~cb).sum()); n10 = int((~ca & cb).sum())
    if n01 + n10 == 0: return 1.0
    return float(stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue)      # exact McNemar
def a12(x, y):
    x, y = np.asarray(x), np.asarray(y); return float(((x[:, None] > y[None, :]).sum() + 0.5 * (x[:, None] == y[None, :]).sum()) / (len(x) * len(y)))
def alpaydin_f(diffs):  # diffs: shape (5,2) of score differences from 5x2 CV
    d = np.asarray(diffs); s2 = ((d - d.mean(1, keepdims=True)) ** 2).sum(1)
    if s2.sum() == 0: return 1.0
    F = (d ** 2).sum() / (2 * s2.sum()); return float(1 - stats.f.cdf(F, 10, 5))
def holm(pvals):
    p = np.asarray(pvals, float); o = np.argsort(p); adj = np.empty_like(p); m = len(p); run = 0
    for r, i in enumerate(o): run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj

In [ ]:
# ---------- 5. BERT for NFR subtypes: hold-out (same split as v2) and 5x2 CV, against the proposed model ----------
rows, cvrows = [], []
for dname, df in [("D1", d1), ("D2", d2)]:
    X, y = task_data(df, "subtype")
    tr, te, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)   # identical to v2
    prop = TfidfModel("linsvc").fit(tr, ytr).predict(te)
    bm = BertClf().fit(tr, ytr); pb = bm.predict(te)
    for mname, p in [(PROPOSED, prop), ("Fine-tuned BERT (adequate budget)", pb)]:
        P, Rr, F, _ = precision_recall_fscore_support(yte, p, average="macro", zero_division=0)
        rows.append({"Dataset": dname, "Model": mname, "n_test": len(yte), "Accuracy": accuracy_score(yte, p), "Macro-P": P, "Macro-R": Rr,
                     "Macro-F1": F, "Weighted-F1": f1_score(yte, p, average="weighted"),
                     "Macro-F1 95% CI": boot_ci(lambda i: f1_score(yte[i], p[i], average="macro"), len(yte)),
                     "Epochs used": getattr(bm, "epochs_used", None) if "BERT" in mname else None})
    rows.append({"Dataset": dname, "Model": "McNemar p (proposed vs BERT)", "Macro-F1": mcnemar(yte, prop, pb)})
    a_, b_ = [], []
    for fold, (a, b) in enumerate(RepeatedStratifiedKFold(n_splits=2, n_repeats=5, random_state=SEED).split(X, y)):  # identical folds to v2
        a_.append(f1_score(y[b], TfidfModel("linsvc").fit(X.iloc[a], y[a]).predict(X.iloc[b]), average="macro"))
        b_.append(f1_score(y[b], BertClf().fit(X.iloc[a], y[a]).predict(X.iloc[b]), average="macro")); print(dname, "fold", fold + 1)
    a_, b_ = np.array(a_), np.array(b_)
    cvrows.append({"Dataset": dname, "Proposed mean macro-F1": a_.mean(), "Proposed SD": a_.std(ddof=1), "BERT mean macro-F1": b_.mean(), "BERT SD": b_.std(ddof=1),
                   "Wilcoxon p": stats.wilcoxon(a_, b_).pvalue if np.any(a_ != b_) else 1.0, "5x2cv F-test p": alpaydin_f((a_ - b_).reshape(5, 2)), "A12 (proposed vs BERT)": a12(a_, b_)})
    R[f"cv_subtype_{dname}"] = {"proposed": a_.tolist(), "bert": b_.tolist()}
save_tab(pd.DataFrame(rows).round(4), "table_bert_subtype_holdout"); save_tab(pd.DataFrame(cvrows).round(4), "table_bert_subtype_cv5x2")
R["bert_subtype_holdout"] = rows; R["bert_subtype_cv"] = cvrows

In [ ]:
# ---------- 6. Link selection: global threshold vs per-FR top-k (D3 gold, D1 silver) ----------
from rank_bm25 import BM25Okapi
def sim_matrices(fr_texts, nfr_texts, fit_texts):
    fr_c = [preprocess(t) for t in fr_texts]; nf_c = [preprocess(t) for t in nfr_texts]
    vec = make_vec().fit([preprocess(t) for t in fit_texts])
    S = {"TF-IDF cosine": (vec.transform(fr_c) @ vec.transform(nf_c).T).toarray()}
    bm = BM25Okapi([d.split() or ["_"] for d in nf_c]); B = np.array([bm.get_scores(q.split()) for q in fr_c])
    S["BM25"] = B / (B.max(1, keepdims=True) + 1e-9)
    if RUN_NEURAL: S["SBERT cosine"] = embed(list(fr_texts)) @ embed(list(nfr_texts)).T
    return S
def prf(acc, G):
    tp = (acc & G).sum(); P_ = tp / max(acc.sum(), 1); R_ = tp / max(G.sum(), 1); return P_, R_, (2 * P_ * R_ / (P_ + R_) if P_ + R_ else 0.0)
def topk_mask(S, k):
    k = min(k, S.shape[1]); idx = np.argsort(-S, axis=1, kind="stable")[:, :k]; M = np.zeros_like(S, dtype=bool)
    np.put_along_axis(M, idx, True, axis=1); return M
def frac_mask(S, f):      # top fraction f of the NFRs for each FR (adapts to project size)
    return topk_mask(S, max(1, int(round(f * S.shape[1]))))
TAUS = np.round(np.arange(0.0, 0.951, 0.025), 3); KS = [1, 2, 3, 5, 8]; FRACS = [0.1, 0.2, 0.3, 0.5]
proj_S = {p: sim_matrices(v["fr"], v["nfr"], promise_nfr[promise_nfr.project != p].text.tolist()) for p, v in D3.items()}
methods = list(proj_S[1].keys())
rules = {"Global τ": (TAUS, lambda S, t: S >= t), "Top-k per FR": (KS, topk_mask), "Top-fraction per FR": (FRACS, frac_mask)}
rows = []
for p in D3:
    G = D3[p]["G"]; others = [q for q in D3 if q != p]
    for mth in methods:
        for rname, (grid, fn) in rules.items():
            best = max(grid, key=lambda g: np.mean([prf(fn(proj_S[q][mth], g), D3[q]["G"])[2] for q in others]))   # tuned on the other 9 projects
            P_, R_, F_ = prf(fn(proj_S[p][mth], best), G)
            rows.append({"Project": p, "Method": mth, "Rule": rname, "Chosen": best, "Pairwise P": P_, "Pairwise R": R_, "Pairwise F1": F_})
    d_ = G.mean(); rows.append({"Project": p, "Method": "—", "Rule": "Accept-all pairs", "Pairwise P": d_, "Pairwise R": 1.0, "Pairwise F1": 2 * d_ / (1 + d_)})
sel = pd.DataFrame(rows); save_tab(sel.round(4), "table_link_selection_D3_by_project")
sel_sum = sel.groupby(["Method", "Rule"])[["Pairwise P", "Pairwise R", "Pairwise F1"]].mean().reset_index()
save_tab(sel_sum.round(4), "table_link_selection_D3_summary"); R["link_selection_D3"] = sel_sum.to_dict(orient="records")
# paired test: best rule per method vs accept-all, per project
sig = []
for mth in methods:
    acc_all = sel[sel.Rule == "Accept-all pairs"].sort_values("Project")["Pairwise F1"].values
    for rname in rules:
        v = sel[(sel.Method == mth) & (sel.Rule == rname)].sort_values("Project")["Pairwise F1"].values
        sig.append({"Method": mth, "Rule": rname, "Mean F1": v.mean(), "Accept-all mean F1": acc_all.mean(),
                    "Wilcoxon p (10 projects)": stats.wilcoxon(v, acc_all).pvalue if np.any(v != acc_all) else 1.0})
save_tab(pd.DataFrame(sig).round(4), "table_link_selection_D3_significance"); R["link_selection_D3_sig"] = sig

# D1 silver labels (same split as v2)
tr1, te1 = train_test_split(d1, test_size=0.2, stratify=d1.type + "|" + d1["sub"], random_state=SEED)
fit1, dev1 = train_test_split(tr1, test_size=0.2, stratify=tr1.type + "|" + tr1["sub"], random_state=SEED)
def d1_part(part, fit_texts):
    fr = part[part.type == "FR"]; nf = part[part.type == "NFR"]
    a = fr.module.to_numpy(dtype=object); b = nf.module.to_numpy(dtype=object)
    return sim_matrices(fr.text.tolist(), nf.text.tolist(), fit_texts), a[:, None] == b[None, :]
Sdev, Gdev = d1_part(dev1, fit1.text.tolist()); Ste, Gte = d1_part(te1, tr1.text.tolist())
rows = []
for mth in methods:
    for rname, (grid, fn) in rules.items():
        if rname == "Top-fraction per FR": grid = [0.005, 0.01, 0.02, 0.05]
        best = max(grid, key=lambda g: prf(fn(Sdev[mth], g), Gdev)[2]); P_, R_, F_ = prf(fn(Ste[mth], best), Gte)
        rows.append({"Method": mth, "Rule": rname, "Chosen": best, "Pairwise P": P_, "Pairwise R": R_, "Pairwise F1": F_})
d_ = Gte.mean(); rows.append({"Method": "—", "Rule": "Accept-all pairs", "Pairwise P": d_, "Pairwise R": 1.0, "Pairwise F1": 2 * d_ / (1 + d_)})
save_tab(pd.DataFrame(rows).round(4), "table_link_selection_D1_silver"); R["link_selection_D1"] = rows

In [ ]:
# ---------- 7. Save ----------
def to_jsonable(o):
    if isinstance(o, dict): return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o
json.dump(to_jsonable(R), open(f"{OUT}/results_v21.json", "w"), indent=1, default=str)
with zipfile.ZipFile("results_v21_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT): z.write(f"{OUT}/{f}", f"results_v21/{f}")
print("Saved results_v21_bundle.zip")
try:
    from google.colab import files; files.download("results_v21_bundle.zip")
except ImportError: pass